In [4]:
class WumpusWorld:
    pass

class KnowledgeAgent:
    pass

In [5]:
from collections import deque
GRID_SIZE = 4


def in_bounds(x, y):
    return 1 <= x <= GRID_SIZE and 1 <= y <= GRID_SIZE


def get_neighbours(cell):
    x, y = cell
    candidates = [(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)]
    return [(nx, ny) for nx, ny in candidates if in_bounds(nx, ny)]


class WumpusWorld:
    def __init__(self):
        self.size = GRID_SIZE
        self.grid = self.create_grid()

        self.agent_start = (1, 1)
        self.agent = self.agent_start

        self.pits = set()
        self.wumpus = (4, 3)
        self.trap = (2, 2)
        self.gold = (4, 4)

    def create_grid(self):
        return [["." for _ in range(self.size)] for _ in range(self.size)]

    def is_pit(self, cell):
        return cell in self.pits

    def is_wumpus(self, cell):
        return cell == self.wumpus

    def is_trap(self, cell):
        return cell == self.trap

    def is_gold(self, cell):
        return cell == self.gold

    def is_empty(self, cell):
        return (
            not self.is_pit(cell)
            and not self.is_wumpus(cell)
            and not self.is_trap(cell)
            and not self.is_gold(cell)
        )

    def get_percepts(self, cell):
        percepts = {
            "breeze": False,
            "stench": False,
            "vibration": False,
            "glitter": False
        }

        for neighbour in get_neighbours(cell):
            if self.is_pit(neighbour):
                percepts["breeze"] = True
            if self.is_wumpus(neighbour):
                percepts["stench"] = True
            if self.is_trap(neighbour):
                percepts["vibration"] = True

        if self.is_gold(cell):
            percepts["glitter"] = True

        print(f"Percepts at {cell}: {percepts}")
        return percepts

    def rebuild_grid(self, agent_position=None):
        self.grid = self.create_grid()

        for pit in self.pits:
            x, y = pit
            self.grid[y - 1][x - 1] = "P"

        if self.wumpus is not None:
            x, y = self.wumpus
            self.grid[y - 1][x - 1] = "W"

        if self.trap is not None:
            x, y = self.trap
            self.grid[y - 1][x - 1] = "T"

        if self.gold is not None:
            x, y = self.gold
            self.grid[y - 1][x - 1] = "G"

        if agent_position is not None:
            x, y = agent_position
            self.grid[y - 1][x - 1] = "A"

    def print_grid(self, agent_position=None):
        self.rebuild_grid(agent_position)

        print("\nCurrent grid:")
        for row in reversed(self.grid):
            print(" ".join(row))
        print()

    def move_wumpus_towards(self, target_cell):
        wx, wy = self.wumpus
        tx, ty = target_cell

        possible_moves = []

        if wx > tx:
            possible_moves.append((wx - 1, wy))
        if wx < tx:
            possible_moves.append((wx + 1, wy))
        if wy > ty:
            possible_moves.append((wx, wy - 1))
        if wy < ty:
            possible_moves.append((wx, wy + 1))

        for new_cell in possible_moves:
            if in_bounds(*new_cell) and not self.is_pit(new_cell) and not self.is_trap(new_cell):
                self.wumpus = new_cell
                print(f"Wumpus moved to {self.wumpus}")
                return

        print("Wumpus could not move.")




class KnowledgeAgent:
    def __init__(self, world):
        self.world = world
        self.position = world.agent_start
        self.alive = True
        self.has_gold = False
        self.escaped = False
        self.stopped = False

        self.visited = set()

        self.safe_from_traps = {self.position}
        self.safe_from_pits = {self.position}
        self.safe_from_wumpus = {self.position}

        self.possible_pits = set()
        self.possible_wumpus = set()
        self.possible_trap = set()

        self.pit_clues = []
        self.wumpus_clues = []
        self.trap_clues = []

        self.known_pits = set()

        self.known_trap = set()
        self.known_wumpus = set()

    def risk_score(self, cell):
          risk = 0

          if cell in self.possible_pits:
              risk += 3

          if cell in self.possible_wumpus:
              risk += 5

          if cell in self.possible_trap:
              risk += 2

          return risk


    def known_safe_cells(self):
      return (
          self.safe_from_pits
          & self.safe_from_wumpus
          & self.safe_from_traps
      ) - self.known_trap - self.known_wumpus


    def find_path(self, start, goals, allowed_cells):## Allows the agent to get to a safe cell via path
      queue = deque()
      queue.append((start, []))

      visited = {start}

      while queue:
          current, path = queue.popleft()

          if current in goals and current != start:
              return path

          for neighbour in get_neighbours(current):
              if neighbour not in visited and neighbour in allowed_cells:
                  visited.add(neighbour)
                  queue.append((neighbour, path + [neighbour]))

      return None

    def perceive(self):
        return self.world.get_percepts(self.position)

    def is_safe(self, cell):
        return (
            cell in self.safe_from_pits
            and cell in self.safe_from_wumpus
            and cell in self.safe_from_traps
        )

    def update_knowledge(self, percepts):
        self.visited.add(self.position)

        self.safe_from_pits.add(self.position)
        self.safe_from_traps.add(self.position)
        self.safe_from_wumpus.add(self.position)

        neighbours = get_neighbours(self.position)

        if not percepts["breeze"]:
            for n in neighbours:
                self.safe_from_pits.add(n)
                self.possible_pits.discard(n)
        else:
            candidates = set()
            for n in neighbours:
                if n not in self.visited and n not in self.safe_from_pits:
                    self.possible_pits.add(n)
                    candidates.add(n)
            if candidates:
              self.pit_clues.append(candidates)

        if not percepts["stench"]:
            for n in neighbours:
                self.safe_from_wumpus.add(n)
                self.possible_wumpus.discard(n)
        else:
          candidates = set()

          for n in neighbours:
              if n not in self.visited and n not in self.safe_from_wumpus:
                  self.possible_wumpus.add(n)
                  candidates.add(n)

          if candidates:
              self.wumpus_clues.append(candidates)

        if not percepts["vibration"]:
            for n in neighbours:
                self.safe_from_traps.add(n)
                self.possible_trap.discard(n)
        else:
          candidates = set()

          for n in neighbours:
              if n not in self.visited and n not in self.safe_from_traps:
                self.possible_trap.add(n)
                candidates.add(n)

          if candidates:
            self.trap_clues.append(candidates)





    def choose_action(self, percepts):
      if percepts["glitter"]:
          return ("grab", None)

      if self.has_gold and self.position == self.world.agent_start:
          return ("exit", None)

      safe_cells = self.known_safe_cells()

      # If agent has gold, find a safe path back to the start
      if self.has_gold:
          path_home = self.find_path(
              self.position,
              {self.world.agent_start},
              safe_cells
          )

          if path_home:
              return ("move", path_home[0])

          return ("stop", None)

      # Try to find the nearest safe unvisited cell
      safe_unvisited = {
          cell for cell in safe_cells
          if cell not in self.visited
      }

      path_to_safe_unvisited = self.find_path(
          self.position,
          safe_unvisited,
          safe_cells
      )

      if path_to_safe_unvisited:
          return ("move", path_to_safe_unvisited[0])

      # If no guaranteed safe route exists, try the least risky unknown neighbour
      neighbours = get_neighbours(self.position)

      unknown_moves = [
          n for n in neighbours
          if n not in self.visited
      ]

      # Prefer unknown cells that are not suspected pits or Wumpus cells
      lower_risk_unknown_moves = [
          n for n in unknown_moves
          if n not in self.possible_pits
          and n not in self.possible_wumpus
          and n not in self.possible_trap
          and n not in self.known_trap
          and n not in self.known_wumpus
      ]

      if lower_risk_unknown_moves:
          best = min(lower_risk_unknown_moves, key=self.risk_score)
          return ("move", best)

      # If every unknown move may contain a pit or Wumpus, stop instead of gambling
      if unknown_moves:
          print("No safe unknown moves available. All unknown moves appear too risky.")
          return ("stop", None)

      return ("stop", None)

    def move_to(self, cell):
        self.position = cell
        self.world.agent = cell

        if self.world.is_pit(cell):
            print("Agent fell into a pit.")
            self.alive = False
            return

        if self.world.is_wumpus(cell):
            print("Agent met the Wumpus.")
            self.alive = False
            return

        if self.world.is_trap(cell):
            print("Agent triggered the trap.")
            self.world.move_wumpus_towards(cell)
            self.known_trap.add(cell)
            self.safe_from_traps.discard(cell)
            self.safe_from_pits.discard(cell)
            self.safe_from_wumpus.discard(cell)
            self.known_wumpus = {}

            if self.world.is_wumpus(cell):
                print("The Wumpus moved onto the agent.")
                self.alive = False
                return

    def step(self):
        if not self.alive or self.escaped:
            return

        percepts = self.perceive()
        self.update_knowledge(percepts)

        action, target = self.choose_action(percepts)

        if action == "grab":
            print("Agent grabbed the gold.")
            self.has_gold = True
            self.world.gold = None

        elif action == "exit":
            print("Agent escaped with the gold.")
            self.escaped = True

        elif action == "move":
            print(f"Agent moves from {self.position} to {target}")
            self.move_to(target)

        elif action == "stop":
            print("Agent stopped because no safe move is known.")
            self.stopped = True

    def run(self, max_steps=20):
        for _ in range(max_steps):
            if not self.alive:
                print("Agent died.")
                break

            if self.escaped or self.stopped:
                print("Run finished.")
                break

            self.world.print_grid(self.position)
            self.step()


def run_test(name, pits, wumpus, trap, gold, max_steps=50):
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    world = WumpusWorld()
    world.pits = set(pits)
    world.wumpus = wumpus
    world.trap = trap
    world.gold = gold

    agent = KnowledgeAgent(world)
    agent.run(max_steps)

    print("\nFinal result:")
    print("Alive:", agent.alive)
    print("Has gold:", agent.has_gold)
    print("Escaped:", agent.escaped)
    print("Stopped:", agent.stopped)
    print("Final position:", agent.position)

# # Example setup
# world = WumpusWorld()

# world.pits = {(3, 1)}
# world.wumpus = (4, 3)
# world.trap = (2, 2)
# world.gold = (4, 4)

# agent = KnowledgeAgent(world)
# agent.run()

def run_trap_test():
    print("\n" + "=" * 60)
    print("TEST 2 - Trap trigger moves the Wumpus, then agent continues")
    print("=" * 60)

    world = WumpusWorld()

    # Test layout
    world.pits = set()
    world.wumpus = (4, 3)
    world.trap = (2, 1)
    world.gold = (4, 4)

    agent = KnowledgeAgent(world)

    print("\nInitial grid:")
    world.print_grid(agent.position)

    print("Initial agent position:", agent.position)
    print("Initial Wumpus position:", world.wumpus)
    print("Trap position:", world.trap)

    print("\nForcing agent to move onto the trap cell...")
    agent.move_to(world.trap)

    print("\nGrid after trap is triggered:")
    world.print_grid(agent.position)

    print("Final agent position after trap:", agent.position)
    print("Wumpus position after trap:", world.wumpus)
    print("Agent alive:", agent.alive)
    print("Known trap cells:", agent.known_trap)

    if world.wumpus != (4, 3):
        print("\nResult: PASS - The Wumpus moved after the trap was triggered.")
    else:
        print("\nResult: FAIL - The Wumpus did not move.")

    if world.trap in agent.known_trap:
        print("Result: PASS - The trap cell was recorded as known unsafe.")
    else:
        print("Result: FAIL - The trap cell was not recorded.")

    print("\nContinuing normal agent behaviour after trap trigger...")
    agent.run(max_steps=20)

    print("\nFinal result after continuing:")
    print("Alive:", agent.alive)
    print("Has gold:", agent.has_gold)
    print("Escaped:", agent.escaped)
    print("Stopped:", agent.stopped)
    print("Final position:", agent.position)

run_test(
    name="Default grid",
     pits={(3, 1)},
    wumpus=(4, 3),
    trap=(2, 2),
    gold=(4, 4)
)


Default grid

Current grid:
. . . G
. . . W
. T . .
A . P .

Percepts at (1, 1): {'breeze': False, 'stench': False, 'vibration': False, 'glitter': False}
Agent moves from (1, 1) to (2, 1)

Current grid:
. . . G
. . . W
. T . .
. A P .

Percepts at (2, 1): {'breeze': True, 'stench': False, 'vibration': True, 'glitter': False}
Agent moves from (2, 1) to (1, 1)

Current grid:
. . . G
. . . W
. T . .
A . P .

Percepts at (1, 1): {'breeze': False, 'stench': False, 'vibration': False, 'glitter': False}
Agent moves from (1, 1) to (1, 2)

Current grid:
. . . G
. . . W
A T . .
. . P .

Percepts at (1, 2): {'breeze': False, 'stench': False, 'vibration': True, 'glitter': False}
No safe unknown moves available. All unknown moves appear too risky.
Agent stopped because no safe move is known.
Run finished.

Final result:
Alive: True
Has gold: False
Escaped: False
Stopped: True
Final position: (1, 2)


In [6]:
run_test(
    name="TEST 1 - Successful gold collection and escape",
    pits={(1, 3)},
    wumpus=(1, 4),
    trap=(2, 3),
    gold=(4, 4)
)


TEST 1 - Successful gold collection and escape

Current grid:
W . . G
P T . .
. . . .
A . . .

Percepts at (1, 1): {'breeze': False, 'stench': False, 'vibration': False, 'glitter': False}
Agent moves from (1, 1) to (2, 1)

Current grid:
W . . G
P T . .
. . . .
. A . .

Percepts at (2, 1): {'breeze': False, 'stench': False, 'vibration': False, 'glitter': False}
Agent moves from (2, 1) to (3, 1)

Current grid:
W . . G
P T . .
. . . .
. . A .

Percepts at (3, 1): {'breeze': False, 'stench': False, 'vibration': False, 'glitter': False}
Agent moves from (3, 1) to (4, 1)

Current grid:
W . . G
P T . .
. . . .
. . . A

Percepts at (4, 1): {'breeze': False, 'stench': False, 'vibration': False, 'glitter': False}
Agent moves from (4, 1) to (4, 2)

Current grid:
W . . G
P T . .
. . . A
. . . .

Percepts at (4, 2): {'breeze': False, 'stench': False, 'vibration': False, 'glitter': False}
Agent moves from (4, 2) to (3, 2)

Current grid:
W . . G
P T . .
. . A .
. . . .

Percepts at (3, 2): {'breeze'